## Introduction

This notebook will test the connections to the various containers and packages needed to complete this workshop.  This includes:

- PostgreSQL
- Senzing
- LanceDB
- NetworkX
- PyVis

The key part is that the final cell should show ✅ for all connections.  

In [ ]:
import importlib
import json
import os
import socket
import time

from IPython.display import IFrame
from IPython.display import HTML

import grpc
import lancedb
import networkx as nx
import pandas as pd
import psycopg2
from pyvis.network import Network

from senzing_grpc import SzAbstractFactoryGrpc
from senzing import SzProduct, SzError

## Test PostgreSQL Connection

Connects to the Postgres database using environment variables set in the Docker Compose file and prints the server version.  A failure here usually means the `erkg_postgres` container is not running or is not yet healthy.

In [ ]:
try:
    conn = psycopg2.connect(
        host=os.getenv('POSTGRES_HOST', 'postgres'),
        port=os.getenv('POSTGRES_PORT', '5432'),
        user=os.getenv('POSTGRES_USER', 'postgres'),
        password=os.getenv('POSTGRES_PASSWORD', 'workshop'),
        database=os.getenv('POSTGRES_DB', 'erkg')
    )
    cursor = conn.cursor()
    cursor.execute('SELECT version();')
    version = cursor.fetchone()
    print(f"✅ PostgreSQL connection successful!")
    print(f"   Version: {version[0][:50]}...")
    cursor.close()
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL connection failed: {e}")

## Check Senzing Network Reachability

Verifies that the Jupyter container can resolve the `senzing` hostname and open a TCP connection to port 8261.  This is a low-level diagnostic confirming network reachability within the container.  If this fails, the problem is the container infrastructure.

In [ ]:
senzing_host = os.getenv('SENZING_GRPC_HOST', 'senzing')
senzing_port = int(os.getenv('SENZING_GRPC_PORT', '8261'))

try:
    # Try to resolve the hostname
    ip_address = socket.gethostbyname(senzing_host)
    print(f"✅ DNS resolution successful: {senzing_host} -> {ip_address}")
    
    # Try to connect to the port
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(5)
    result = sock.connect_ex((senzing_host, senzing_port))
    sock.close()
    
    if result == 0:
        print(f"✅ Senzing service is reachable at {senzing_host}:{senzing_port}")
    else:
        print(f"❌ Senzing service is not responding on port {senzing_port}")
        print(f"   Make sure the senzing container is running:")
        print(f"   Run: docker-compose ps")
        
except socket.gaierror as e:
    print(f"❌ Cannot resolve hostname '{senzing_host}'")
    print(f"   Error: {e}")
    print(f"\n   Troubleshooting steps:")
    print(f"   1. Check all containers are running: docker-compose ps")
    print(f"   2. Check they're on the same network: docker network inspect odsc_east_2026_erkg-network")
    print(f"   3. Restart services: docker-compose restart")
except Exception as e:
    print(f"❌ Connection test failed: {e}")

## Test Senzing gRPC Connection

This confirms the Senzing application is actually running, initialized, and responding to SDK calls.  Opens a Google Remote Procedure Call (gRPC) the Senzing service and calls `get_version()` to confirm the SDK is reachable and responding.  gRPC (Google Remote Procedure Call) is a framework that lets one program call functions in another program running on a different machine, as if they were local function calls.  It is necessary because Jupyter and Senzing are in different containers, so gRPC bridges this gap between the two.

In [ ]:
try:
    # Create gRPC channel with options
    grpc_host = os.getenv('SENZING_GRPC_HOST', 'senzing')
    grpc_port = os.getenv('SENZING_GRPC_PORT', '8261')
    grpc_url = f"{grpc_host}:{grpc_port}"
    
    print(f"   Connecting to: {grpc_url}")
    
    # Create channel with explicit options
    options = [
        ('grpc.enable_http_proxy', 0),
        ('grpc.max_receive_message_length', 100 * 1024 * 1024),  # 100MB
    ]
    channel = grpc.insecure_channel(grpc_url, options=options)
    
    # Wait for channel to be ready (with timeout)
    print("   Waiting for channel to be ready...")
    grpc.channel_ready_future(channel).result(timeout=10)
    
    # Create Senzing factory using the channel
    sz_factory = SzAbstractFactoryGrpc(channel)
    
    # Test SzProduct to get version info
    sz_product = sz_factory.create_product()
    version_info = sz_product.get_version()
    print(f"✅ Senzing gRPC connection successful!")
    print(f"   Connected to: {grpc_url}")
    print(f"   Senzing version info:\n{version_info[:200]}...")
    
except grpc.FutureTimeoutError:
    print(f"❌ Connection timeout - Senzing service not responding")
    print(f"   Check if senzing container is running: docker-compose ps senzing")
except grpc.RpcError as e:
    print(f"❌ gRPC Error: {e.code()}")
    print(f"   Details: {e.details()}")
    if e.code() == grpc.StatusCode.UNAVAILABLE:
        print(f"\n   The senzing service is not available. Please check:")
        print(f"   1. Is the container running? docker-compose ps")
        print(f"   2. Are all services on the same network?")
        print(f"   3. Try restarting: docker-compose restart senzing")
except SzError as e:
    print(f"❌ Senzing SDK error: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {type(e).__name__}: {e}")

## Check Senzing License

Retrieves and displays the Senzing license details, including the expiration date and record limit.  The evaluation license we are using today allows up to 500 records, which is plenty for this workshop.

In [ ]:
try:
    grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'senzing')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
    channel = grpc.insecure_channel(grpc_url)
    sz_factory = SzAbstractFactoryGrpc(channel)
    
    sz_product = sz_factory.create_product()
    license_info = sz_product.get_license()
    print(f"✅ Senzing license verified")
    print(f"   License info:\n{license_info[:200]}...")
    
except SzError as e:
    print(f"❌ License check failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

## Configure Data Sources

In order to load data into Senzing you need to register data sources for it.  This cell registers three test data sources (CUSTOMERS, REFERENCE, WATCHLIST) that will be used in the next notebook into the Senzing configuration if they do not already exist.  The updated config is saved and set as the new default so Senzing can accept records from these sources.

In [ ]:
data_sources = ["CUSTOMERS", "REFERENCE", "WATCHLIST"]

try:
    grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'senzing')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
    channel = grpc.insecure_channel(grpc_url)
    sz_factory = SzAbstractFactoryGrpc(channel)
    
    sz_configmgr = sz_factory.create_configmanager()
    sz_config = sz_configmgr.create_config_from_template()
    
    # Check existing data sources
    existing_sources = json.loads(sz_config.get_data_source_registry())
    existing_codes = {ds['DSRC_CODE'] for ds in existing_sources.get('DATA_SOURCES', [])}
    
    # Add new data sources
    added_count = 0
    for ds_code in data_sources:
        if ds_code not in existing_codes:
            result = sz_config.register_data_source(ds_code)
            print(f"✅ Added data source: {ds_code}")
            added_count += 1
        else:
            print(f"ℹ️  Data source already exists: {ds_code}")
    
    if added_count > 0:
        # Save configuration
        config_definition = sz_config.export()
        config_id = sz_configmgr.register_config(
            config_definition,
            "Workshop configuration with test data sources"
        )
        sz_configmgr.set_default_config_id(config_id)
        
        # Reinitialize with new config
        sz_factory.reinitialize(config_id)
        
        print(f"\n✅ Configuration saved with ID: {config_id}")
    
    print("✅ Data sources configured successfully")
    
except SzError as e:
    print(f"❌ Data source configuration failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

## Add a Test Record

As a brief demonstration and continued test of the Senzing connection, this cell adds a single test record for "Robert Smith" into the CUSTOMERS data source.  It then confirms that the Senzing engine can accept and process an incoming record.

In [ ]:
test_record = {
    "DATA_SOURCE": "CUSTOMERS",
    "RECORD_ID": "TEST_001",
    "NAME_FULL": "Robert Smith",
    "ADDR_FULL": "123 Main Street, Las Vegas NV 89102",
    "PHONE_NUMBER": "702-555-1212",
    "DATE_OF_BIRTH": "1980-01-15"
}

try:
    grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'senzing')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
    channel = grpc.insecure_channel(grpc_url)
    sz_factory = SzAbstractFactoryGrpc(channel)
    sz_engine = sz_factory.create_engine()
    
    # Add a record
    result = sz_engine.add_record(
        test_record["DATA_SOURCE"],
        test_record["RECORD_ID"],
        json.dumps(test_record)
    )
    print(f"✅ Record added successfully")
    print(f"   Data Source: {test_record['DATA_SOURCE']}")
    print(f"   Record ID: {test_record['RECORD_ID']}")
    print(f"   Name: {test_record['NAME_FULL']}")
    
except SzError as e:
    print(f"❌ Record addition failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

## Retrieve Entity by Record ID

As a further test, this cell looks up the resolved entity that Senzing created from the test record and prints its entity ID and name.  This shows how Senzing links raw records to resolved entities.

In [ ]:
try:
    grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'senzing')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
    channel = grpc.insecure_channel(grpc_url)
    sz_factory = SzAbstractFactoryGrpc(channel)
    sz_engine = sz_factory.create_engine()
    
    # Retrieve the entity
    entity = sz_engine.get_entity_by_record_id("CUSTOMERS", "TEST_001")
    entity_data = json.loads(entity)
    
    entity_id = entity_data['RESOLVED_ENTITY']['ENTITY_ID']
    entity_name = entity_data['RESOLVED_ENTITY']['ENTITY_NAME']
    
    print(f"✅ Entity retrieved successfully")
    print(f"   Entity ID: {entity_id}")
    print(f"   Entity Name: {entity_name}")
    
    # Display record count
    records = entity_data['RESOLVED_ENTITY']['RECORDS']
    print(f"   Number of records in entity: {len(records)}")
    
except SzError as e:
    print(f"❌ Entity retrieval failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

## Search by Attributes

As one final test and brief demonstration, this cell searches Senzing for entities matching a name and address, then prints how many results were returned.  

In [ ]:
try:
    grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'senzing')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
    channel = grpc.insecure_channel(grpc_url)
    sz_factory = SzAbstractFactoryGrpc(channel)
    sz_engine = sz_factory.create_engine()
    
    # Search by attributes
    search_attributes = {
        "NAME_FIRST": "Robert",
        "NAME_LAST": "Smith",
        "ADDR_FULL": "123 Main St, Las Vegas NV"
    }
    
    search_result = sz_engine.search_by_attributes(json.dumps(search_attributes))
    search_data = json.loads(search_result)
    
    num_results = len(search_data.get('RESOLVED_ENTITIES', []))
    print(f"✅ Search completed successfully")
    print(f"   Search attributes: {search_attributes}")
    print(f"   Found {num_results} matching entities")
    
    # Display first result if available
    if num_results > 0:
        first_result = search_data['RESOLVED_ENTITIES'][0]
        print(f"\n   First result:")
        print(f"   - Entity ID: {first_result['ENTITY']['RESOLVED_ENTITY']['ENTITY_ID']}")
        print(f"   - Match Score: {first_result.get('MATCH_SCORE', 'N/A')}")
    
except SzError as e:
    print(f"❌ Search failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

## Clean Up Test Record

Finally, we remove this record from the database so we begin clean in the next notebook. 

In [ ]:
try:
    grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'senzing')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
    channel = grpc.insecure_channel(grpc_url)
    sz_factory = SzAbstractFactoryGrpc(channel)
    sz_engine = sz_factory.create_engine()
    
    # Delete test record
    sz_engine.delete_record("CUSTOMERS", "TEST_001")
    print("✅ Test record deleted successfully")
    print("   Data Source: CUSTOMERS")
    print("   Record ID: TEST_001")
    
except SzError as e:
    print(f"❌ Record deletion failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

## Test LanceDB

We will use LanceDB for our vector database.  This cell creates a small test table in LanceDB, runs a vector similarity query, then drops the table.  This confirms that the vector store is mounted and working before we use it to store entity embeddings later in the workshop.

In [ ]:
try:
    # Create a test database
    db_path = "/workspace/lancedb_data/test_db"
    db = lancedb.connect(db_path)
    
    # Create a simple test table
    test_data = pd.DataFrame({
        "id": [1, 2, 3],
        "text": ["hello", "world", "test"],
        "vector": [[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]]
    })
    
    table = db.create_table("test_table", test_data, mode="overwrite")
    print(f"✅ LanceDB initialized successfully!")
    print(f"   Database path: {db_path}")
    print(f"   Created test table with {len(table)} rows")
    
    # Verify we can query
    results = table.search([0.2, 0.3]).limit(2).to_pandas()
    print(f"   Query test: Retrieved {len(results)} rows")
    
    # Clean up
    db.drop_table("test_table")
    print("✅ Test table cleaned up")
    
except Exception as e:
    print(f"❌ LanceDB test failed: {e}")

## Test NetworkX

We will be creating our knowledge graphs in memory using NetworkX.  This cell builds a small four-node graph and computes basic metrics like degree and shortest path.  This is a quick heck that the NetworkX package is installed and functional.

In [ ]:
try:
    # Create a simple test graph
    G = nx.Graph()
    G.add_edges_from([(1, 2), (2, 3), (3, 4), (4, 1)])
    
    # Calculate some basic metrics
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    avg_degree = sum(dict(G.degree()).values()) / num_nodes
    
    print(f"✅ NetworkX working!")
    print(f"   Created test graph:")
    print(f"   - Nodes: {num_nodes}")
    print(f"   - Edges: {num_edges}")
    print(f"   - Average degree: {avg_degree:.2f}")
    
    # Test shortest path
    path = nx.shortest_path(G, 1, 3)
    print(f"   - Shortest path 1→3: {path}")
    
except Exception as e:
    print(f"❌ NetworkX test failed: {e}")

## Test PyVis

We will be visualizing some of our graphs using PyVis.  So this cell renders a simple three-node interactive graph inline in the notebook.  Make sure you can see the graph and drag the nodes around.

In [ ]:
try:
    # Create network
    net = Network(height="400px", width="100%", notebook=True, cdn_resources='in_line')
    net.add_node(1, label="Node 1", color="#97C2FC", size=25)
    net.add_node(2, label="Node 2", color="#FFAB91", size=25)
    net.add_node(3, label="Node 3", color="#A5D6A7", size=25)
    net.add_edge(1, 2)
    net.add_edge(2, 3)
    net.add_edge(1, 3)
    
    # Save to current directory
    test_file = "pyvis_test.html"
    net.save_graph(test_file)
    
    print(f"✅ PyVis working!")
    print(f"   Created visualization with 3 nodes, 3 edges")
    print(f"   Interactive graph below - you can drag nodes!\n")
    
    # Read the HTML file and display it
    with open(test_file, 'r', encoding='utf-8') as f:
        html_string = f.read()
    
    # Display the HTML inline
    display(HTML(html_string))
    
    # Clean up
    os.remove(test_file)
    
except Exception as e:
    print(f"❌ PyVis test failed: {e}")

## Verify All Workshop Packages

Iterates through every required Python package and reports whether it is installed.  Run this last to quickly see what is missing.

In [ ]:
print("Checking all workshop Python packages...")
print()

packages = {
    'senzing_grpc': 'Senzing gRPC Client',
    'senzing': 'Senzing SDK',
    'psycopg2': 'PostgreSQL Driver',
    'lancedb': 'LanceDB Vector Store',
    'pandas': 'Pandas Data Analysis',
    'networkx': 'NetworkX Graph Analysis',
    'pyvis': 'PyVis Graph Visualization',
    'dotenv': 'Python DotEnv'
}

all_good = True
for pkg, description in packages.items():
    try:
        importlib.import_module(pkg)
        print(f"✅ {description:30s} ({pkg})")
    except ImportError:
        print(f"❌ {description:30s} ({pkg}) - NOT INSTALLED")
        all_good = False

print()
if all_good:
    print("✅ All required packages installed!")
else:
    print("❌ Some packages are missing. Check Dockerfile and rebuild.")